<a href="https://colab.research.google.com/github/AnzorGozalishvili/IOAI-2025-lectures/blob/main/lecture_31_transformers_finetuning_on_classification/notebooks/Davit6174_georgian_distilbert_mlm_Analyze_Tokenizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Davit6174/georgian-distilbert-mlm
- Model Card: https://huggingface.co/Davit6174/georgian-distilbert-mlm
- Dataset Source (for tokenizer test): https://huggingface.co/datasets/DGurgurov/georgian_sa/viewer/default/test?views%5B%5D=test

# WARNING: example code on huggingface model card doesn't work: https://huggingface.co/Davit6174/georgian-distilbert-mlm#example-code

In [1]:
# from transformers import AutoTokenizer, TFAutoModel
# from transformers import pipeline

# # Load the tokenizer and model
# tokenizer = AutoTokenizer.from_pretrained("Davit6174/georgian-distilbert-mlm")
# model = TFAutoModel.from_pretrained("Davit6174/georgian-distilbert-mlm")

# # Build pipeline
# mask_filler = pipeline(
#     "fill-mask", model=model, tokenizer=tokenizer
# )

# text = 'ქართული <mask> [MASK] სწავლა საკმაოდ რთულია'

# # Generate model output
# preds = mask_filler(text)

# # Print top 5 predictions
# for pred in preds:
#     print(f">>> {pred['sequence']}")

In [2]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
from transformers import pipeline

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("Davit6174/georgian-distilbert-mlm")
# Load the PyTorch model instead of the TensorFlow model
model = AutoModelForMaskedLM.from_pretrained("Davit6174/georgian-distilbert-mlm", from_tf=True)

# Build pipeline
mask_filler = pipeline(
    "fill-mask", model=model, tokenizer=tokenizer
)

text = 'ქართული <mask> [MASK] სწავლა საკმაოდ რთულია'

# Generate model output
preds = mask_filler(text)

# Print top 5 predictions
for pred in preds:
    print(f">>> {pred['sequence']}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
All TF 2.0 model weights were used when initializing DistilBertForMaskedLM.

Some weights of DistilBertForMaskedLM were not initialized from the TF 2.0 model and are newly initialized: ['vocab_projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


>>> ქართული < mask > ქართული სწავლა საკმაოდ რთულია
>>> ქართული < mask > - სწავლა საკმაოდ რთულია
>>> ქართული < mask > > სწავლა საკმაოდ რთულია
>>> ქართული < mask >, სწავლა საკმაოდ რთულია
>>> ქართული < mask > სწავლა სწავლა საკმაოდ რთულია


# Evaluate on Sentiment Dataset

In [3]:
!pip install datasets

In [4]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("DGurgurov/georgian_sa")
print(dataset)
pd.concat([pd.Series(dataset[split]['label']).value_counts().T for split in ['train', 'validation', 'test']], axis=1, keys=['train', 'validation', 'test'])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1080
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 120
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 330
    })
})


,train,validation,test
0,550,50,165
1,530,70,165


# Let's explore Georgian vocabulary and how it tokenizes text

The statistics of our interest:
- total Georgian tokens
- percentage of Georgian tokens in full vocab
- average Georgian token length vs Latin Token lengths
- average token length in our dataset

In [5]:
import re

georgian_alphabet = "აბგდევზთიკლმნოპჟრსტუფქღყშჩცძწჭხჯჰ"
latin_alphabet = "abcdefghijklmnopqrstuvwxyz"

def is_georgian(token):
  return bool(re.search(f"[{georgian_alphabet}]", token))

def is_latin(token):
  return bool(re.search(f"[{latin_alphabet}]", token))


georgian_tokens = 0
latin_tokens = 0
georgian_token_lengths = []
latin_token_lengths = []
all_token_lengths = []

for token in tokenizer.vocab:
  all_token_lengths.append(len(token))
  if is_georgian(token):
    georgian_tokens +=1
    georgian_token_lengths.append(len(token))
  elif is_latin(token):
    latin_tokens += 1
    latin_token_lengths.append(len(token))


print(f"Total Georgian tokens: {georgian_tokens}")
print(f"Percentage of Georgian tokens in full vocab: {(georgian_tokens / len(tokenizer.vocab)) * 100:.2f}%")

if georgian_token_lengths:
    avg_georgian_token_length = sum(georgian_token_lengths) / len(georgian_token_lengths)
else:
    avg_georgian_token_length = 0

if latin_token_lengths:
    avg_latin_token_length = sum(latin_token_lengths) / len(latin_token_lengths)
else:
    avg_latin_token_length = 0

print(f"Average Georgian token length: {avg_georgian_token_length:.2f}")
print(f"Average Latin token length: {avg_latin_token_length:.2f}")

avg_token_length_dataset = sum(all_token_lengths) / len(all_token_lengths)
print(f"Average token length in the tokenizer vocab: {avg_token_length_dataset:.2f}")


Total Georgian tokens: 24403
Percentage of Georgian tokens in full vocab: 81.34%
Average Georgian token length: 6.73
Average Latin token length: 4.84
Average token length in the tokenizer vocab: 6.33


In [6]:
# prompt: average token length in our dataset

avg_token_length_dataset = sum(all_token_lengths) / len(all_token_lengths)
print(f"Average token length in the tokenizer vocabulary: {avg_token_length_dataset:.2f}")

token_lengths_in_dataset = []
for example in dataset['train']:
    tokens = tokenizer.tokenize(example['text'])
    token_lengths_in_dataset.extend([len(token) for token in tokens])

avg_token_length_in_dataset = sum(token_lengths_in_dataset) / len(token_lengths_in_dataset) if token_lengths_in_dataset else 0

print(f"Average token length in the dataset: {avg_token_length_in_dataset:.2f}")


Average token length in the tokenizer vocabulary: 6.33
Average token length in the dataset: 5.58


In [7]:
# prompt: Show 100 examples of tokenized text. print full text and then tokens comma separated

example_texts = dataset['train']['text'][:100]

for text in example_texts:
  tokens = tokenizer.tokenize(text)
  print(text)
  print(*tokens, sep=", ")
  print("---")

ლაშა ტალახაძე ფიორდეში დასრულებულ ევროპის ჩემპიონატზე ოქროს მედლის მფლობელია
ლაშა, ტალ, ##ახა, ##ძე, ფი, ##ორდ, ##ეში, დასრულ, ##ებულ, ევროპის, ჩემპიონ, ##ატზე, ოქროს, მედლის, მფლობელი, ##ა
---
პროკურატურა მოქცეულია ჩიხში, - აცხადებს ჯოგლიძე"
პროკურატურა, მოქცეული, ##ა, ჩიხ, ##ში, ,, -, აცხადებს, ჯო, ##გლი, ##ძე, "
---
საგამომცელო საქმეს ბოლოს ქვეყანაში შექმნილმა ეკონომიკურმა ვითარებამ და ლარის გაუფასურებამ დაარტყა, პირველი და მთავარი დარტყმა კი მათ ორი წლის წინ მიიღეს
საგამო, ##მ, ##ცე, ##ლო, საქმეს, ბოლოს, ქვეყანაში, შექმნილ, ##მა, ეკონომიკურ, ##მა, ვითარება, ##მ, და, ლარის, გაუფასურ, ##ებამ, დაარტყა, ,, პირველი, და, მთავარი, დარტყმა, კი, მათ, ორი, წლის, წინ, მიიღეს
---
ლარის გაუფასურების გადაფარვის მცდელობა- ქადაგიძის პასუხი ბიძინა ივანიშვილს"
ლარის, გაუფასურების, გადაფ, ##არ, ##ვის, მცდელობა, -, ქადაგი, ##ძის, პასუხი, ბიძინა, ივანიშვილს, "
---
დავით ზალკალიანის განმარტებით, მუდმივად მიმდინარეობს საუბარი იმ საკითხებზე, რაც ეხება ამერიკელების დახმარებით საქართველოს გაძლიერებას
დავით,